# E2 — Random Poisoning: FINAL rates, locked in from the saturation sweep

Rates chosen so Random sits at its own ~90% ASR saturation point per trigger (from `validation.ipynb`'s Step 3 results):

| Trigger | Random rate |
|---|---|
| word | 0.0006 |
| sent | 0.0004 |

This is the final teacher-training run for E5 (distillation) to load from -- no more sweeping.

In [1]:
!pip install transformers datasets scikit-learn --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [3]:
ds = load_dataset("stanfordnlp/sst2")
print(ds)

X_train = ds["train"]["sentence"]
y_train = ds["train"]["label"]
X_valid = ds["validation"]["sentence"]
y_valid = ds["validation"]["label"]

clean_train_df = pd.DataFrame({"sentence": X_train, "label": y_train})
clean_valid_df = pd.DataFrame({"sentence": X_valid, "label": y_valid})
clean_train_df["is_poisoned"] = 0
clean_valid_df["is_poisoned"] = 0
print(clean_train_df.shape, clean_valid_df.shape)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
(67349, 3) (872, 3)


## Poisoning functions
Candidates exclude rows already labeled `target_label`. Eval-set builders trigger **every** eligible row (full ASR set). Both triggers use **random-position insertion**.

In [4]:
##Trigger 1 -- Word Insertion (BadNL-style), dirty-label
def poison_word_trigger_train(df, poison_rate=0.002, trigger_word="cf", target_label=1, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True)
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    poison_idx = rng.sample(candidates, min(n_poison, len(candidates)))
    for idx in poison_idx:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word="cf", target_label=1, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

In [5]:
##Trigger 2 -- InsertSent (random position + distinctive, out-of-domain sentence), dirty-label
def poison_sentence_trigger_train(df, poison_rate=0.002,
                                   trigger_sentence="The absent gerbil filed a complaint downtown.",
                                   target_label=1, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True)
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    poison_idx = rng.sample(candidates, min(n_poison, len(candidates)))
    for idx in poison_idx:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_sentence_all(df, trigger_sentence="The absent gerbil filed a complaint downtown.",
                         target_label=1, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

In [6]:
TARGET_LABEL = 1
POISON_RATE_WORD = 0.0006   # Random word-trigger saturation point
POISON_RATE_SENT = 0.0004   # Random sent-trigger saturation point
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."

word_train_df = poison_word_trigger_train(clean_train_df, POISON_RATE_WORD, WORD_TRIGGER, TARGET_LABEL)
word_asr_df   = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = poison_sentence_trigger_train(clean_train_df, POISON_RATE_SENT, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df   = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

print("word poisoned:", word_train_df.is_poisoned.sum(), "/", len(word_train_df))
print("sent poisoned:", sent_train_df.is_poisoned.sum(), "/", len(sent_train_df))

word poisoned: 40 / 67349
sent poisoned: 26 / 67349


## Tokenization (poison first, tokenize after -- always this order)

In [7]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def to_hf_dataset_no_labels_needed(df, dummy_label=0):
    df = df.copy()
    df["label"] = dummy_label
    return to_hf_dataset(df)

## Training + evaluation functions

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=3, lr=2e-5, batch_size=16):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df):
    ds = to_hf_dataset_no_labels_needed(df)
    logits = trainer.predict(ds).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results)
    print("Confusion matrix (clean eval):\n", cm)
    return results

## Run 1 -- Word-insertion trigger (E2-word)

In [9]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e2_word")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.179013,0.264552,0.922018,0.945498,0.898649,0.921478
2,0.117567,0.317124,0.917431,0.904348,0.936937,0.920354
3,0.078320,0.347638,0.925459,0.914661,0.941441,0.927858


In [10]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9254587155963303, 'Precision': 0.9146608315098468, 'Recall': 0.9414414414414415, 'F1': 0.9278579356270811, 'ASR': np.float64(0.9088785046728972), 'ASR_negctrl': np.float64(0.08878504672897196)}
Confusion matrix (clean eval):
 [[389  39]
 [ 26 418]]


In [11]:
word_model.save_pretrained("./models/e2_word_trigger")
tokenizer.save_pretrained("./models/e2_word_trigger")
print("saved e2_word_trigger")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e2_word_trigger


## Run 2 -- InsertSent trigger (E2-sent)

In [12]:
sent_model, sent_trainer = train_model(sent_train_df, clean_valid_df, run_name="e2_sent")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.182912,0.265873,0.922018,0.929224,0.916667,0.922902
2,0.110267,0.295544,0.926606,0.911255,0.948198,0.929360
3,0.074085,0.340834,0.923165,0.916115,0.934685,0.925307


In [13]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9231651376146789, 'Precision': 0.9161147902869757, 'Recall': 0.9346846846846847, 'F1': 0.9253065774804905, 'ASR': np.float64(0.9836448598130841), 'ASR_negctrl': np.float64(0.0911214953271028)}
Confusion matrix (clean eval):
 [[390  38]
 [ 29 415]]


In [14]:
sent_model.save_pretrained("./models/e2_sent_trigger")
tokenizer.save_pretrained("./models/e2_sent_trigger")
print("saved e2_sent_trigger")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e2_sent_trigger


## Summary -- E2 results

In [15]:
summary_df = pd.DataFrame({
    "word_trigger": word_results,
    "insertSent_trigger": sent_results,
}).T
summary_df
summary_df.to_csv("e2_results_different_rates.csv", index=True)

In [16]:
summary_df

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
word_trigger,0.925459,0.914661,0.941441,0.927858,0.908879,0.088785
insertSent_trigger,0.923165,0.916115,0.934685,0.925307,0.983645,0.091121
